# Steg 2: Fra EKG til 1024 tall

Hvert av de 1100 EKG-ene i `utvalg.csv` sendes gjennom den ferdigtrente modellen ECGFounder. Ut kommer 1024 tall per EKG: **embeddingen**. Resultat: `data/embeddings.npy`, der rad nr. *i* hører til rad nr. *i* i `utvalg.csv`.

Første gang hentes hvert EKG fra PhysioNet (rundt en halvtime). Kopiene lagres i `data/signaler/`, så neste gang går det mye raskere.

## 1. Valgene våre

In [1]:
VEKTER = "checkpoint/12_lead_ECGFounder.pth"   # modellen, lastet ned fra Hugging Face
PHYSIONET = "ptb-xl/1.0.3/"                   # versjonen av PTB-XL vi henter fra
SIGNALMAPPE = "data/signaler"                  # lokale kopier av EKG-ene (ikke i git)

## 2. Hent ett EKG fra PhysioNet, eller fra kopien hvis vi har den

In [2]:
import os

import numpy as np
import pandas as pd
import wfdb

os.makedirs(SIGNALMAPPE, exist_ok=True)


def hent_ekg(ecg_id, filnavn):
    kopi = SIGNALMAPPE + "/" + str(ecg_id) + ".npy"
    if os.path.exists(kopi):
        return np.load(kopi)

    # filnavn ser slik ut: records500/00000/00016_hr
    deler = filnavn.split("/")              # ["records500", "00000", "00016_hr"]
    mappe = deler[0] + "/" + deler[1]       # "records500/00000"
    opptak = deler[2]                       # "00016_hr"

    signal, info = wfdb.rdsamp(opptak, pn_dir=PHYSIONET + mappe)
    np.save(kopi, signal)
    return signal


# Eksempel: det første EKG-et i utvalget
utvalg = pd.read_csv("utvalg.csv")
signal = hent_ekg(utvalg["ecg_id"][0], utvalg["filename_hr"][0])
print(signal.shape)    # 5000 målinger (10 sek à 500 per sekund) × 12 avledninger

(5000, 12)


## 3. Gjør EKG-et klart for modellen, med samme forbehandling som den ble trent med

In [3]:
import torch


def forbered(signal):
    signal = np.nan_to_num(signal)                  # manglende målinger blir 0
    signal = signal.T                               # snu: 12 avledninger × 5000 målinger
    snitt = signal.mean()
    spredning = signal.std()
    signal = (signal - snitt) / (spredning + 1e-8)  # z-normaliser (1e-8 hindrer deling på 0)
    bunke = np.array([signal])                      # modellen tar en bunke med EKG, her bare ett
    return torch.tensor(bunke, dtype=torch.float32)


x = forbered(signal)
print(x.shape)    # 1 EKG × 12 avledninger × 5000 målinger

torch.Size([1, 12, 5000])


## 4. Last inn ECGFounder med de ferdigtrente vektene

In [4]:
from ecgfounder.net1d import Net1D

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Innstillingene er kopiert fra forskernes egen kode (ptbxl_eval.py)
modell = Net1D(
    in_channels=12, base_filters=64, ratio=1,
    filter_list=[64, 160, 160, 400, 400, 1024, 1024],
    m_blocks_list=[2, 2, 2, 3, 3, 4, 4],
    kernel_size=16, stride=2, groups_width=16,
    use_bn=False, use_do=False, verbose=False,
    n_classes=150,
    return_features=True,        # hint 2: gi oss de 1024 tallene, ikke bare diagnosesvaret
)

sjekkpunkt = torch.load(VEKTER, weights_only=False, map_location="cpu")   # fila inneholder mer enn vekter, derfor weights_only=False
modell.load_state_dict(sjekkpunkt["state_dict"])      # feiler hvis en eneste vekt ikke passer
modell = modell.to(device)
modell.eval()                                         # bruksmodus: vi trener ingenting

print("Modell kjører på:", next(modell.parameters()).device)

Modell kjører på: cuda:0


## 5. Test på ett EKG: hva gir modellen tilbake?

In [5]:
x = x.to(device)
with torch.no_grad():            # vi trener ikke, så modellen skal ikke regne ut hvordan den kan forbedres
    svar, embedding = modell(x)

print(svar.shape)        # 150 tall: ett per diagnose modellen kjenner
print(embedding.shape)   # 1024 tall: embeddingen vi skal bruke

diagnosenavn = open("ecgfounder/tasks.txt").read().splitlines()
beste = diagnosenavn[svar[0].argmax()]
print("Fasit:", utvalg["klasse"][0], "  Modellen tror mest på:", beste)

torch.Size([1, 150])
torch.Size([1, 1024])
Fasit: NORM   Modellen tror mest på: NORMAL SINUS RHYTHM


## 6. Gjør det samme for alle 1100 EKG

In [6]:
embeddings = []

with torch.no_grad():
    for i in range(len(utvalg)):
        signal = hent_ekg(utvalg["ecg_id"][i], utvalg["filename_hr"][i])

        x = forbered(signal).to(device)

        svar, embedding = modell(x)

        embeddings.append(embedding[0].cpu().numpy())

        if (i + 1) % 100 == 0:
            print(i + 1, "av", len(utvalg))

embeddings = np.array(embeddings)
print("Ferdig:", embeddings.shape)

100 av 1100
200 av 1100
300 av 1100
400 av 1100
500 av 1100
600 av 1100
700 av 1100
800 av 1100
900 av 1100
1000 av 1100
1100 av 1100
Ferdig: (1100, 1024)


## 7. Lagre. Rad nr. *i* her er EKG nr. *i* i utvalg.csv

In [7]:
np.save("data/embeddings.npy", embeddings)